In [8]:
#import necessary package
import torch
import copy
import torch.nn.utils.prune as prune
from torchvision import transforms, datasets, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [9]:
# preprocess images
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # MobileNetV2 default input size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])  # ImageNet normalization
])

train_dataset = datasets.ImageFolder("../data/dataset/Training", transform=transform)
test_dataset   = datasets.ImageFolder("../data/dataset/Test", transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
# Load model
model = torch.load("fruit_mobilenetv2.pth")

# Training SetUp
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [10]:
# evaluate function
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# calculate sensitivity of each layer
def measure_layer_sensitivity(model, layer_name, test_loader, device):
    temp_model = copy.deepcopy(model)
    temp_model.to(device)
    temp_model.eval()

    # find the layer
    module = dict(temp_model.named_modules())[layer_name]

    # prune 10% of filters
    prune.ln_structured(module, name="weight", amount=0.1, n=1, dim=0)
    prune.remove(module, "weight")

    # evaluate accuracy
    acc = evaluate(temp_model,test_loader, device)

    return acc


In [11]:
sensitivities = {}
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        print(f"Estimate sensitivity of layer: {name}")
        acc = measure_layer_sensitivity(model, name, test_loader, device)
        print(f"Validation Accuracy: {acc:.2f}%")
        sensitivities[name] = acc

Estimate sensitivity of layer: features.0.0
Validation Accuracy: 89.35%
Estimate sensitivity of layer: features.1.conv.0.0
Validation Accuracy: 89.35%
Estimate sensitivity of layer: features.1.conv.1
Validation Accuracy: 42.97%
Estimate sensitivity of layer: features.2.conv.0.0
Validation Accuracy: 87.17%
Estimate sensitivity of layer: features.2.conv.1.0
Validation Accuracy: 88.05%
Estimate sensitivity of layer: features.2.conv.2
Validation Accuracy: 71.56%
Estimate sensitivity of layer: features.3.conv.0.0
Validation Accuracy: 87.42%
Estimate sensitivity of layer: features.3.conv.1.0
Validation Accuracy: 87.21%
Estimate sensitivity of layer: features.3.conv.2
Validation Accuracy: 78.88%
Estimate sensitivity of layer: features.4.conv.0.0
Validation Accuracy: 89.37%
Estimate sensitivity of layer: features.4.conv.1.0
Validation Accuracy: 65.81%
Estimate sensitivity of layer: features.4.conv.2
Validation Accuracy: 81.73%
Estimate sensitivity of layer: features.5.conv.0.0
Validation Accur

In [12]:
# Use structured pruning: Assign pruning ratio based on sensitivity
originalModel_acc = evaluate(model, test_loader, device)

pruning_plan = {}

for layer, acc in sensitivities.items():
    drop = originalModel_acc - acc

    if drop < 0.5:
        pruning_plan[layer] = 0.5   # prune 50%
    elif drop < 1.0:
        pruning_plan[layer] = 0.3   # prune 30%
    elif drop < 2.0:
        pruning_plan[layer] = 0.1   # prune 10%
    else:
        pruning_plan[layer] = 0.0   # too sensitive, skip

# Apply pruning plan into model
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        amount = pruning_plan[name]
        if amount > 0:
            prune.ln_structured(module, name="weight", amount=amount, n=1, dim=0)
            prune.remove(module, "weight")
            print(f"Pruned {amount*100:.0f}% of filters in {name}")


Pruned 50% of filters in features.0.0
Pruned 50% of filters in features.1.conv.0.0
Pruned 10% of filters in features.2.conv.1.0
Pruned 10% of filters in features.3.conv.0.0
Pruned 50% of filters in features.4.conv.0.0
Pruned 50% of filters in features.5.conv.0.0
Pruned 30% of filters in features.5.conv.2
Pruned 50% of filters in features.6.conv.0.0
Pruned 50% of filters in features.6.conv.1.0
Pruned 30% of filters in features.6.conv.2
Pruned 50% of filters in features.7.conv.0.0
Pruned 10% of filters in features.7.conv.2
Pruned 50% of filters in features.8.conv.0.0
Pruned 50% of filters in features.8.conv.2
Pruned 50% of filters in features.9.conv.0.0
Pruned 10% of filters in features.9.conv.1.0
Pruned 10% of filters in features.10.conv.0.0
Pruned 10% of filters in features.10.conv.1.0
Pruned 10% of filters in features.10.conv.2
Pruned 50% of filters in features.11.conv.0.0
Pruned 50% of filters in features.11.conv.2
Pruned 30% of filters in features.12.conv.0.0
Pruned 50% of filters i

In [13]:
# fine-tune the pruned model
for epoch in range(5):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}, Test Accuracy: {acc:.2f}%")





Epoch 1, Test Accuracy: 79.70%
Epoch 2, Test Accuracy: 79.54%
Epoch 3, Test Accuracy: 78.60%
Epoch 4, Test Accuracy: 79.02%
Epoch 5, Test Accuracy: 78.76%


In [14]:
torch.save(model, "../model/prunned_fruit_mobilenetv2.pth")

CAT-DOG CLASSIFICATION

In [1]:
# Standard library
import copy
import glob
import multiprocessing
import os
import time
import zipfile

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import torch.nn.utils.prune as prune

# Related third party
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
def print_model_size(mdl):
    torch.save(mdl.state_dict(), "tmp.pt")
    print("%.2f MB" %(os.path.getsize("tmp.pt")/1e6))
    os.remove('tmp.pt')

In [4]:
input_size = (224,224)
mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize(input_size),  # Resize to a fixed size
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [5]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label, folder_name in enumerate(['Dog', 'Cat']):
            folder_path = os.path.join(self.root_dir, folder_name)
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                
                try:
                    with Image.open(file_path) as img:
                        
                        if img.mode != 'RGB':
                            img = img.convert('RGB')
                        
                        if img.mode != 'RGB':
                            print(f"Skipping {file_path} because it does not have 3 channels (RGB)")
                            continue

                        self.image_paths.append(file_path)
                        self.labels.append(label)
                        
                except Exception as e:
                    print(f"Skipping {file_path} due to error: {e}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        with Image.open(image_path) as img:

            if img.mode != 'RGB':
                img = img.convert('RGB')

            if self.transform:
                img = self.transform(img)
            
        return img, label

In [6]:
dataset = CustomDataset(root_dir='../data/PetImages', transform=transform)

# Calculate split sizes
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Split dataset into train and test
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

Skipping ../data/PetImages\Dog\11702.jpg due to error: cannot identify image file '../data/PetImages\\Dog\\11702.jpg'


c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\lib\site-packages\PIL\TiffImagePlugin.py:864: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Skipping ../data/PetImages\Dog\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Dog\\Thumbs.db'
Skipping ../data/PetImages\Cat\666.jpg due to error: cannot identify image file '../data/PetImages\\Cat\\666.jpg'
Skipping ../data/PetImages\Cat\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Cat\\Thumbs.db'


In [7]:
# Create DataLoader for train and test sets
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([128, 3, 224, 224])
torch.Size([128])


In [8]:
def train_epoch(model, criterion, optimizer, data_loader, device,epoch):
    model.train()
    
    epoch_loss = 0.0
    num_batches = len(data_loader)
    
    for batch_idx, (image, target) in enumerate(tqdm(data_loader)):
        image, target = image.to(device), target.to(device)
        
        output = model(image)
        
        loss = criterion(output, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    print(f"Epoch = {epoch+1} || Training Loss: {avg_epoch_loss:.4f}")


def evaluate(model, criterion, data_loader, device,epoch):
    
    model.eval()
    
    epoch_loss = 0.0
    
    correct_predictions = 0
    total_predictions = 0
    
    num_batches = len(data_loader)
    
    with torch.no_grad():
       
        for image, target in tqdm(data_loader):
            image, target = image.to(device), target.to(device)
            output = model(image)
            loss = criterion(output, target)
            # Accumulate batch loss
            epoch_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(output, 1)  # Get the predicted class index
            correct_predictions += (predicted == target).sum().item()
            total_predictions += target.size(0)
            
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    accuracy = correct_predictions / total_predictions
    
    print(f"Epoch = {epoch+1} || Validation Loss: {avg_epoch_loss:.4f} || Validation Accuracy: {accuracy:.4f}")
    return accuracy*100

# calculate sensitivity of each layer
def measure_layer_sensitivity(model, layer_name, test_loader, device, criterion, epoch):
    temp_model = copy.deepcopy(model)
    temp_model.to(device)
    temp_model.eval()

    # find the layer
    module = dict(temp_model.named_modules())[layer_name]

    # prune 10% of filters
    prune.ln_structured(module, name="weight", amount=0.1, n=1, dim=0)
    prune.remove(module, "weight")

    # evaluate accuracy
    acc = evaluate(temp_model, criterion, test_loader, device, epoch)

    return acc


In [9]:
class MobileNet(torch.nn.Module):
    def __init__(self):
        super(MobileNet, self).__init__()
        self.model = models.mobilenet_v2(weights=None)  
        
        # for param in self.model.parameters():
        #     param.requires_grad = False
            
        
        
        self.model.classifier[1] = nn.Sequential(
            nn.Linear(in_features=self.model.classifier[1].in_features,out_features=512),
            nn.LeakyReLU(negative_slope=0.02,inplace=False),
            nn.BatchNorm1d(num_features=512),
            nn.Dropout(p=0.4,inplace=False),
            nn.Linear(in_features=512,out_features=2),
            nn.Softmax(dim=1))
        
        # print(self.model)

    def forward(self, x):
        x = self.model(x)
        return x

In [10]:
model = MobileNet()
model.load_state_dict(torch.load("../model/original_catndog_mobilenetv2.pth"))
PrunningModel = copy.deepcopy(model)
print_model_size(PrunningModel)

11.76 MB


In [11]:
# Training Setting
num_epochs = 15
PrunningModel = PrunningModel.to(device)
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(PrunningModel.parameters(), lr = 0.0001)

In [12]:
sensitivities = {}
for name, module in PrunningModel.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        print(f"Estimate sensitivity of layer: {name}")
        acc = measure_layer_sensitivity(PrunningModel, name, test_loader, device, criterion=criterion, epoch=0)
        print(f"Validation Accuracy: {acc:.2f}%")
        sensitivities[name] = acc

Estimate sensitivity of layer: model.features.0.0


100%|██████████| 40/40 [02:23<00:00,  3.59s/it]


Epoch = 1 || Validation Loss: 0.3162 || Validation Accuracy: 0.9968
Validation Accuracy: 99.68%
Estimate sensitivity of layer: model.features.1.conv.0.0


100%|██████████| 40/40 [02:31<00:00,  3.79s/it]


Epoch = 1 || Validation Loss: 0.3162 || Validation Accuracy: 0.9968
Validation Accuracy: 99.68%
Estimate sensitivity of layer: model.features.1.conv.1


...

Epoch = 1 || Validation Loss: 0.3161 || Validation Accuracy: 0.9970
Validation Accuracy: 99.70%
Estimate sensitivity of layer: model.features.17.conv.2


100%|██████████| 40/40 [02:24<00:00,  3.60s/it]


Epoch = 1 || Validation Loss: 0.3170 || Validation Accuracy: 0.9962
Validation Accuracy: 99.62%
Estimate sensitivity of layer: model.features.18.0


100%|██████████| 40/40 [02:24<00:00,  3.61s/it]

Epoch = 1 || Validation Loss: 0.3162 || Validation Accuracy: 0.9972
Validation Accuracy: 99.72%


In [13]:
# Use structured pruning: Assign pruning ratio based on sensitivity
originalModel_acc = evaluate(PrunningModel, criterion, test_loader, device, epoch=0)

pruning_plan = {}

for layer, acc in sensitivities.items():
    drop = originalModel_acc - acc

    if drop < 0.5:
        pruning_plan[layer] = 0.5   # prune 50%
    elif drop < 1.0:
        pruning_plan[layer] = 0.3   # prune 30%
    elif drop < 2.0:
        pruning_plan[layer] = 0.1   # prune 10%
    else:
        pruning_plan[layer] = 0.0   # too sensitive, skip

# Apply pruning plan into model
for name, module in PrunningModel.named_modules():
    if isinstance(module, nn.Conv2d):
        amount = pruning_plan[name]
        if amount > 0:
            prune.ln_structured(module, name="weight", amount=amount, n=1, dim=0)
            # prune.remove(module, "weight")
            print(f"Pruned {amount*100:.0f}% of filters in {name}")


100%|██████████| 40/40 [02:18<00:00,  3.47s/it]

Epoch = 1 || Validation Loss: 0.3162 || Validation Accuracy: 0.9968
Pruned 50% of filters in model.features.0.0
Pruned 50% of filters in model.features.1.conv.0.0
Pruned 50% of filters in model.features.2.conv.0.0
Pruned 50% of filters in model.features.2.conv.1.0
Pruned 30% of filters in model.features.2.conv.2
Pruned 50% of filters in model.features.3.conv.0.0
Pruned 50% of filters in model.features.3.conv.1.0
Pruned 50% of filters in model.features.3.conv.2
Pruned 50% of filters in model.features.4.conv.0.0
Pruned 50% of filters in model.features.4.conv.1.0
Pruned 30% of filters in model.features.4.conv.2
Pruned 50% of filters in model.features.5.conv.0.0
Pruned 50% of filters in model.features.5.conv.2
Pruned 50% of filters in model.features.6.conv.0.0
Pruned 50% of filters in model.features.6.conv.1.0
Pruned 50% of filters in model.features.6.conv.2
Pruned 50% of filters in model.features.7.conv.0.0
Pruned 10% of filters in model.features.7.conv.1.0
Pruned 50% of filters in model.

In [14]:
# Train with Pruning masks - Fine-Tuning
for nepoch in range(num_epochs):
    train_epoch(PrunningModel, criterion, optimizer, train_loader, device, nepoch)
    PrunningModel.eval()
    torch.save(PrunningModel.state_dict(), f"../model/beforeRemovePruning_catndog_mobilenetv2_epoch{nepoch}.pth")

100%|██████████| 157/157 [27:58<00:00, 10.69s/it]


Epoch = 1 || Training Loss: 0.5835


100%|██████████| 157/157 [25:42<00:00,  9.83s/it]


Epoch = 2 || Training Loss: 0.4987


100%|██████████| 157/157 [25:09<00:00,  9.62s/it]


...

Epoch = 13 || Training Loss: 0.3630


100%|██████████| 157/157 [30:06<00:00, 11.51s/it]


Epoch = 14 || Training Loss: 0.3595


100%|██████████| 157/157 [28:44<00:00, 10.98s/it]

Epoch = 15 || Training Loss: 0.3559


In [15]:
# Permanently apply pruning to the weights, remove wrapper and mask system
for name, module in PrunningModel.named_modules():
    if isinstance(module, nn.Conv2d):
        try:
            prune.remove(module, "weight")
            print(f"Remove Pruning from {name}")
        except:
            pass

# Save the pruned model
print("Evaluating pruned model...")
PrunningModel.eval()
evaluate(PrunningModel,criterion, test_loader,torch.device("cpu"),nepoch)
torch.save(PrunningModel.state_dict(), "../model/finalSensitivePruning_catndog_mobilenetv2.pth")

Remove Pruning from model.features.0.0
Remove Pruning from model.features.1.conv.0.0
Remove Pruning from model.features.2.conv.0.0
Remove Pruning from model.features.2.conv.1.0
Remove Pruning from model.features.2.conv.2
Remove Pruning from model.features.3.conv.0.0
Remove Pruning from model.features.3.conv.1.0
Remove Pruning from model.features.3.conv.2
Remove Pruning from model.features.4.conv.0.0
Remove Pruning from model.features.4.conv.1.0
Remove Pruning from model.features.4.conv.2
Remove Pruning from model.features.5.conv.0.0
Remove Pruning from model.features.5.conv.2
Remove Pruning from model.features.6.conv.0.0
Remove Pruning from model.features.6.conv.1.0
Remove Pruning from model.features.6.conv.2
Remove Pruning from model.features.7.conv.0.0
Remove Pruning from model.features.7.conv.1.0
Remove Pruning from model.features.7.conv.2
Remove Pruning from model.features.8.conv.0.0
Remove Pruning from model.features.8.conv.1.0
Remove Pruning from model.features.8.conv.2
Remove Pr

100%|██████████| 40/40 [02:22<00:00,  3.57s/it]

Epoch = 15 || Validation Loss: 0.4332 || Validation Accuracy: 0.8818
